# 01 — Berth Allocation

**HYPAQ Ports · Initial educational reference · Apache-2.0**

Assign two synthetic vessels to two berths and two one-slot service windows.
Minimize total waiting time. Each vessel must be assigned once, no earlier than
arrival, to a compatible, available berth; a berth cannot serve two vessels in
one slot. Inputs and assumptions are documented in `data/synthetic/README.md`.

**Implemented:** classical exact enumeration, an eight-bit QUBO, and an Ising
energy-equivalence check. **Planned:** CP-SAT, Ket/KBW QAOA and a timeline.
The exhaustive QUBO sweep below is classical and does not simulate a quantum circuit.

Run cells in order. For Colab, first follow `notebooks/README.md` to clone shared
code and data into the runtime. No third-party Python package is needed by these cells.

In [1]:
from pathlib import Path
import sys
import json

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src/hypaq_ports/reference.py").is_file()
             and (p / "data/synthetic").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook inside the repository; see notebooks/README.md for Colab setup.")
sys.path.insert(0, str(ROOT / "src"))
from hypaq_ports.reference import (
    load_json, solve_berth_exact, berth_metrics, validate_berth_schedule,
    build_berth_qubo, qubo_energy, decode_berth_bits, qubo_to_ising,
    solve_crane_exact, evaluate_crane_schedule, apply_arrival_delay,
)
print("Repository:", ROOT.name)


Repository: hypaq-ports-starter


## 1. Load the instance and solve the operational model

The input uses slots 0 and 1, arrivals 0 and 1, and unit service duration. The solver enumerates all permitted assignment combinations and reports an optimum only after finishing the search.

In [2]:
instance = load_json(ROOT / "data/synthetic/berth_tiny_v1.json")
reference = solve_berth_exact(instance)
print(json.dumps({"input": instance, "reference": reference}, indent=2))
assert reference["status"] == "OPTIMAL"
assert reference["metrics"]["total_waiting_slots"] == 0
assert validate_berth_schedule(instance, reference["schedule"])["feasible"]


{
  "input": {
    "instance_id": "berth_tiny_v1",
    "schema_version": "0.1",
    "provenance": "Handcrafted educational synthetic data; not a real terminal",
    "time_unit": "abstract_slot",
    "slot_duration_minutes": 60,
    "berths": [
      "B1",
      "B2"
    ],
    "slots": [
      0,
      1
    ],
    "vessels": [
      {
        "id": "V1",
        "arrival_slot": 0,
        "service_slots": 1,
        "compatible_berths": [
          "B1",
          "B2"
        ]
      },
      {
        "id": "V2",
        "arrival_slot": 1,
        "service_slots": 1,
        "compatible_berths": [
          "B1",
          "B2"
        ]
      }
    ],
    "unavailable": []
  },
  "reference": {
    "status": "OPTIMAL",
    "method": "exhaustive_enumeration",
    "schedule": [
      {
        "vessel": "V1",
        "berth": "B1",
        "slot": 0
      },
      {
        "vessel": "V2",
        "berth": "B1",
        "slot": 1
      }
    ],
    "metrics": {
      "feasible": true

## 2. Build and exhaustively evaluate the QUBO

There is one bit per `(vessel, berth, slot)`, for eight bits total.

The objective adds waiting cost to three penalties: exactly one choice per vessel,
forbidden assignments, and simultaneous use of a berth by different vessels.
The default penalty exceeds a bound on any feasible waiting cost. The QUBO
uses an upper-triangular coefficient dictionary, with each pair counted once.
The full energy includes a constant offset.

In [3]:
import itertools
model = build_berth_qubo(instance)
sweep = [(qubo_energy(model, bits), bits)
         for bits in itertools.product((0, 1), repeat=len(model["variables"]))]
minimum = min(energy for energy, bits in sweep)
ground_states = [bits for energy, bits in sweep if energy == minimum]
for bits in ground_states:
    metrics = berth_metrics(instance, decode_berth_bits(model, bits))
    assert metrics["feasible"]
    assert metrics["total_waiting_slots"] == reference["score"]
print(json.dumps({"binary_variables": len(model["variables"]),
                  "bitstrings_evaluated": len(sweep), "penalty": model["penalty"],
                  "offset": model["offset"], "minimum_energy": minimum,
                  "number_of_ground_states": len(ground_states),
                  "example_ground_schedule": decode_berth_bits(model, ground_states[0])}, indent=2))


{
  "binary_variables": 8,
  "bitstrings_evaluated": 256,
  "penalty": 2,
  "offset": 4,
  "minimum_energy": 0,
  "number_of_ground_states": 4,
  "example_ground_schedule": [
    {
      "vessel": "V1",
      "berth": "B2",
      "slot": 0
    },
    {
      "vessel": "V2",
      "berth": "B2",
      "slot": 1
    }
  ]
}


## 3. Verify the Ising mapping

Use `x_i=(1-z_i)/2` with `z_i` in `{+1,-1}`. Preserve the constant offset when
comparing `E_QUBO(x)` and `E_Ising(z)`. This is a classical algebraic check over
all 256 assignments. Mapping these coefficients to a Ket Hamiltonian is future work.

In [4]:
ising = qubo_to_ising(model)
errors = []
for qubo_value, bits in sweep:
    spins = [1 - 2 * bit for bit in bits]
    ising_value = ising["constant"] + sum(h * z for h, z in zip(ising["h"], spins))
    ising_value += sum(j * spins[i] * spins[k] for (i, k), j in ising["j"].items())
    errors.append(abs(qubo_value - ising_value))
print("Maximum energy mismatch:", max(errors))
assert max(errors) < 1e-10


Maximum energy mismatch: 0.0


## Explore next

1. Restrict a vessel to a single berth and rerun the model and QUBO sweep.
2. Add an unavailable berth/slot and check every new ground state independently.
3. Create an infeasible instance and explain why a minimum QUBO energy does not guarantee feasibility.
4. Extend the time horizon carefully: enumeration scales exponentially.
5. Later, add CP-SAT and Ket/KBW adapters using the same operational validator.

**Limitations:** one-slot services, synthetic data, no channel, tide, draft,
crane coordination, uncertainty model or operational integration.